# Hour 2 — File Areas, Folders & Files

Recap: in Hour 1 you created a `DaluxClient` and explored projects, companies and users. This hour is about the
document-management side of Dalux Build: **file areas** (top-level document libraries) → **folders** → **files**,
plus downloading and uploading content.

**By the end of this hour you will be able to:**

- List file areas and resolve one by name
- Browse folders, either one page at a time or fully paginated
- Resolve a human-readable path like `"Files/4_Design/C07_Geometry"` straight to a folder ID
- List files and filter/download them in bulk
- Upload a new file using the chunked upload flow


## 0. Reconnect

Same setup as Hour 1 — each notebook is self-contained so you can run them independently.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

# Works whether Jupyter was launched from the repo root or from tutorials/
for candidate in (Path(".env"), Path("../.env")):
    if candidate.exists():
        load_dotenv(candidate)
        break
else:
    load_dotenv()  # fall back to variables already exported in the shell

import os
assert os.getenv("DALUX_API_KEY"), "DALUX_API_KEY not found — copy .env.example to .env and fill it in"
assert os.getenv("DALUX_BASE_URL"), "DALUX_BASE_URL not found — copy .env.example to .env and fill it in"
print("DALUX_BASE_URL:", os.getenv("DALUX_BASE_URL"))

In [ ]:
from dalux_build import create_client

dalux = create_client()
dalux

In [ ]:
# Pick the first project in your account to work with for the rest of this notebook.
# Swap this for dalux.projects.get_project_by_name("Your Project Name") if you want a specific one.
projects_response = dalux.projects.list_projects()

PROJECT_ID = projects_response[0].project_id
print("Using project:", projects_response[0].project_name, f"({PROJECT_ID})")

dalux.set_default_project(PROJECT_ID) # This enables you to omit the `project_id` parameter in future calls, for convenience. You can always override it by passing a different `project_id` explicitly.

## 1. File areas — `dalux.file_areas`

A **file area** is a top-level document library on a project (e.g. "Files", "Published", "Shared"). Most file/folder
operations need both a `project_id` and a `file_area_id`.

A File area is like if you had multiple Google Drive or SharePoint document libraries for a single project, each with its own folder structure.

In [ ]:
import pandas as pd

file_areas_response = dalux.file_areas.get_file_areas() # Add `to_dataframe=True` to get a Pandas DataFrame instead of a list of Pydantic models.
file_areas_response

In [ ]:
FILE_AREA_ID = file_areas_response[0].file_area_id
FILE_AREA_NAME = file_areas_response[0].file_area_name
print("Using file area:", FILE_AREA_NAME, f"({FILE_AREA_ID})")

# Convenience helper if you know the name instead of the ID:
same_id = dalux.file_areas.get_file_area_by_name(FILE_AREA_NAME) # FILE_AREA_NAME can be "Files", "Published files", "Shared files". Be aware that the file area names are always in English, even if you see them in another language in the Dalux Build web app. 

# You can always set the default file area on the client for convenience, just like you did with the project:
dalux.set_default_file_area(FILE_AREA_ID) # This enables you to omit the `file_area_id` parameter in future calls, for convenience. You can always override it by passing a different `file_area_id` explicitly.

## 2. Folders — `dalux.folders`

Two ways to list folders, same pattern you'll see again for files and tasks:

- `list_folders(...)` — **one page** as returned by the API (fast, but you may need to follow `nextPage` yourself)
- `get_all_folders(...)` — follows the API's bookmark-based pagination **automatically** and returns every folder
  as a flat `List[Folder]`

Under the hood, `get_all_folders` calls the shared `dalux_build.utils.pagination.paginate()` helper, which keeps
requesting pages using the `bookmark` query param from each response's `nextPage` link until none is left.

In [ ]:
one_page = dalux.folders.list_folders(full_response=True) # Add `to_dataframe=True` to get a Pandas DataFrame instead of a list of Pydantic models.
print(f"First page: {len(one_page.items)} folder(s), metadata: {one_page.metadata}")

In [ ]:
all_folders = dalux.folders.get_all_folders(verbose=True)
print(f"\nTotal folders (all pages): {len(all_folders)}")

### Resolving a path directly

Instead of manually walking `parent_folder_id` relationships, you can resolve a full, human-readable path in one
call. Paths start with the file area name. `get_folder_by_path` even prints its resolution steps when
`verbose=True`.

In [ ]:
# Replace this with a real folder path from FILE_AREA_NAME in your project, e.g.
# f"{FILE_AREA_NAME}/Drawings/Structural"
example_path = "Files/4_Design"  # falls back to just the file area root if you haven't picked a subfolder yet
# ! Be aware that the path should always start with the file area name.

folder = dalux.folders.get_folder_by_path(example_path, verbose=True)
folder

## 3. Files — `dalux.files`

Same one-page-vs-all-pages pattern as folders. `get_all_files_in_folder` additionally accepts either a
`(file_area_id, folder_id)` pair *or* a full path string.

In [ ]:
files_response = dalux.files.list_files()
print(f"First page: {len(files_response)} file(s)")

all_files = dalux.files.get_all_files(verbose=True, to_dataframe=True)
all_files.head()

In [ ]:
# Files inside one specific folder, resolved by ID:
if all_folders:
    files_in_folder = dalux.files.get_all_files_in_folder(
        all_folders[0].folder_id,
        verbose=True,
    )
    print(f"{len(files_in_folder)} file(s) in folder {all_folders[0].folder_name!r}")

## 4. Building & browsing a full folder tree

`get_file_area_tree` assembles folders (and, if you pass `files_api=dalux.files`, files too) into one nested tree
in a single call — fetching folders and files **concurrently** in two threads. Each node looks like:

```python
{"id": ..., "name": ..., "path": "parent/child/name", "children": [...], "files": [...]}
```


In [ ]:
tree = dalux.folders.get_file_area_tree(files_api=dalux.files, verbose=True)

def print_tree(node, depth=0, max_depth=2):
    label = node["name"] if node["path"] == "" else node["path"].split("/")[-1]
    print("  " * depth + f"📁 {label}  ({len(node['files'])} file(s))")
    if depth >= max_depth:
        return
    for child in node["children"]:
        print_tree(child, depth + 1, max_depth)

print_tree(tree)

In [ ]:
# Get a subtree for a specific branch of the file area tree

tree_keys = [node["path"] for node in tree["children"]]
tree_keys

In [ ]:
branch_key = tree_keys[1] if tree_keys else None
if branch_key:
    subtree = tree["children"][1]  # or use a helper function to find the node by path
    print(f"\nSubtree for branch {branch_key!r}:")
    print_tree(subtree, max_depth=2)

In [ ]:
# Display all the files in the subtree, if any using tree and do it recursively for all children of the subtree.
def list_files_in_subtree(node):
    files = node["files"]
    for child in node["children"]:
        files.extend(list_files_in_subtree(child))
    return files

files_in_subtree = list_files_in_subtree(subtree)
print(f"\nTotal files in subtree {branch_key!r}: {len(files_in_subtree)}")

## 5. Downloading files

Three complementary helpers, all under `dalux.files`:

- `get_file(project_id, path, download=True, save_path=...)` — a **single** file by full path
- `bulk_download_folder(...)` — every file in a folder, optionally filtered by name/extension via `FileNameFilter`
- `bulk_download_files(project_id, [paths_or_ids], ...)` — an explicit list of files

All three skip re-downloading a file if an identical local copy (same revision) already exists, and can optionally
write a sidecar `.json`-ish `.txt` metadata file next to each download (`save_metadata=True`).

In [ ]:
DOWNLOAD_DIR = "downloads"

# Single file by path (skip if you don't have a real file path handy)
# result = dalux.files.get_file(
#     PROJECT_ID, f"{FILE_AREA_NAME}/Drawings/Structural/plan.pdf",
#     download=True, save_path=DOWNLOAD_DIR, verbose=True,
# )

In [ ]:
# Search for a folder by name

SEARCH_FOLDER_NAME = "C07.11_Sketch" # Modify this to a folder name that exists in your project. The search is case-insensitive and will return the first match found.

folders_found = [folder for folder in all_folders if folder.folder_name.lower() == SEARCH_FOLDER_NAME.lower()]

folders_found

In [ ]:
# Take folder id from the search result if unique, otherwise pick the first one. You can also use the folder path instead of the name if you know it.
if len(folders_found) == 1:
    folder_id_to_download = folders_found[0].folder_id
elif len(folders_found) > 1:
    print(f"Multiple folders found with name {SEARCH_FOLDER_NAME!r}, using the first one.")
    folder_id_to_download = folders_found[0].folder_id
else:
    print(f"No folders found with name {SEARCH_FOLDER_NAME!r}.")
    folder_id_to_download = None

In [ ]:
from dalux_build.models import FileNameFilter

# Bulk-download every IFC in the file area's root folder that matches a name filter.
filters = FileNameFilter(extensions=[".ifc"]) 

downloaded = dalux.files.bulk_download_folder(
    folder_id=folder_id_to_download,
    save_path=DOWNLOAD_DIR,
    filters=filters,
    verbose=True,
)
print(f"Downloaded {len(downloaded)} file(s) to ./{DOWNLOAD_DIR}/")

## 6. File revisions — `dalux.file_revisions`

Every file upload creates a new revision. `get_file_revision_content` fetches the raw content of one specific
revision — useful for comparing versions or archiving history (a `file.file_revision_id` comes from a `File`
object you already fetched).

In [ ]:
if all_files:
    sample_file = all_files[0]
    if sample_file.file_revision_id:
        content = dalux.file_revisions.get_file_revision_content(
            PROJECT_ID, FILE_AREA_ID, sample_file.file_id, sample_file.file_revision_id
        )
        print(type(content), str(content)[:200])

## 7. Uploading a file (chunked)

> ⚠️ **This cell writes real data to your Dalux project.** It's disabled by default — flip `RUN_UPLOAD_DEMO` to
> `True` only once you have a real, disposable test folder to upload into.

Uploading is a three-step flow (there's no single "upload this file" convenience call — you compose the three
primitives yourself):

1. `create_upload(project_id, file_area_id, {"fileName": ..., "mimeType": ...})` → returns an `uploadGuid`
2. `upload_file_part(project_id, file_area_id, upload_guid, chunk_bytes)` → called once per chunk (call it
   repeatedly for large files; a single call is enough for small ones)
3. `finish_upload(project_id, file_area_id, upload_guid, {"folderId": ...})` → finalizes and returns the new
   `fileId`

In [ ]:
RUN_UPLOAD_DEMO = False  # set to True (and pick a real TARGET_FOLDER_ID) to actually upload

if RUN_UPLOAD_DEMO:
    import pathlib

    local_path = pathlib.Path("hello_dalux.txt")
    local_path.write_text("Uploaded from the Hour 2 tutorial notebook.\n")

    TARGET_FOLDER_ID = all_folders[0].folder_id if all_folders else None

    upload = dalux.file_upload.create_upload(
        PROJECT_ID, FILE_AREA_ID,
        {"fileName": local_path.name, "mimeType": "text/plain"},
    )
    upload_guid = upload["uploadGuid"]

    CHUNK_SIZE = 5 * 1024 * 1024  # 5 MB; loop over chunks for larger files
    with open(local_path, "rb") as f:
        while chunk := f.read(CHUNK_SIZE):
            dalux.file_upload.upload_file_part(PROJECT_ID, FILE_AREA_ID, upload_guid, chunk)

    result = dalux.file_upload.finish_upload(
        PROJECT_ID, FILE_AREA_ID, upload_guid,
        {"folderId": TARGET_FOLDER_ID},
    )
    print("New file ID:", result["fileId"])
else:
    print("Upload demo skipped (RUN_UPLOAD_DEMO is False).")

## Recap & what's next

You can now browse file areas, folders and files (one page at a time or fully paginated), resolve human-readable
paths directly to IDs, build a full tree in one call, download files individually or in filtered bulk, and upload
new files through the chunked upload flow.

**Next up — Hour 3:** tasks, forms, work packages, a quick tour of the remaining read-only resources, the
lower-level `paginate` / `find_by_field` utilities you can reuse for endpoints not yet wrapped, and a small
capstone project status report.
